# 07 合并、连接与重塑

真实数据永远散在多张表里：订单表、客户表、商品表、城市表……
分析之前要先把它们拼起来；分析过程中又常常要在"宽表"和"长表"之间来回换。

本节两件事：

1. **拼表**：`merge`（按列连接）、`join`（按索引连接）、`concat`（上下/左右堆叠）；
2. **变形**：`melt`（宽 → 长）、`pivot`（长 → 宽）、`stack` / `unstack`。

> 环境：`llm_sft`，从 `python/pandas` 目录启动。

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_columns", 20)

DATA = Path("data")
if not DATA.exists():
    DATA = Path("python/pandas/data")

customers = pd.read_csv(DATA / "customers.csv")
orders = pd.read_csv(DATA / "customer_orders.csv", parse_dates=["order_date"])
print("客户表:", customers.shape, "| 订单表:", orders.shape)
customers.head(3)

客户表: (8, 5) | 订单表: (10, 5)


,customer_id,name,city,level,signup_date
0,C01,北京晨光科技,北京,金卡,2022-03-01
1,C02,上海蓝鲸贸易,上海,银卡,2022-07-15
2,C03,广州南风服饰,广州,普通,2023-01-09


In [2]:
orders.head(3)

,order_id,customer_id,order_date,category,amount
0,T001,C01,2024-01-08,数码,2598.0
1,T002,C02,2024-01-15,家居,199.5
2,T003,C01,2024-02-11,数码,1299.0


## 1. `merge`：按"列"连接

四种连接方式，记住"**保留哪边的键**"就不会错：

| how | 保留的键 | 典型的业务问题 |
| --- | --- | --- |
| `inner`（默认） | 两边都有的键 | 只分析"下过单的客户" |
| `left` | 左表的全部键 | 所有客户，包括没下单的 |
| `right` | 右表的全部键 | 所有订单，包括客户信息缺失的 |
| `outer` | 两边的并集 | 查数据缺口（谁没匹配上） |

对不上的位置一律补 `NaN`，**不会报错也不会丢行**——这就是最容易出错的地方。

In [3]:
for how in ["inner", "left", "right", "outer"]:
    merged = customers.merge(orders, on="customer_id", how=how)
    print(f"{how:6s} 行数={len(merged):2d}  客户数={merged['customer_id'].nunique()}")

inner  行数= 9  客户数=6
left   行数=11  客户数=8
right  行数=10  客户数=7
outer  行数=12  客户数=9


In [4]:
# indicator=True 会多一列 _merge，直接告诉你每行来自哪边
outer = customers.merge(orders, on="customer_id", how="outer", indicator=True)
print(outer["_merge"].value_counts().to_dict())
outer[outer["_merge"] != "both"][["customer_id", "name", "order_id", "amount", "_merge"]]

{'both': 9, 'left_only': 2, 'right_only': 1}


,customer_id,name,order_id,amount,_merge
8,C06,成都锦里食品,NaN,NaN,left_only
10,C08,上海浦江物流,NaN,NaN,left_only
11,C09,NaN,T005,88.0,right_only


In [5]:
# validate 检查"键在某一侧是否唯一"，防止行数悄悄翻倍
try:
    customers.merge(orders, on="customer_id", validate="one_to_one")
except pd.errors.MergeError as e:
    print("validate 拦截:", str(e)[:100])

customers.merge(orders, on="customer_id", validate="one_to_many").shape

validate 拦截: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
 customer_id


(9, 9)

## 2. 键的三种隐形坑

1. **列名不同**：用 `left_on` / `right_on`；
2. **类型不同**：一边 `"C01"`（字符串），一边 `1`（整数）→ merge 直接报错；
3. **值不同**：`"C01"` vs `"C1"`、前后空格、大小写、全角字符 → **不报错，只是全部匹配不上**。

第 3 种最阴险：结果看起来"跑了"，其实全是 NaN。所以拼完表一定要检查匹配率。

In [6]:
customers.dtypes[["customer_id"]]

customer_id    str
dtype: object

In [7]:
# 坑 A：键的类型不同 → merge 直接报错。这是"幸运"的情况，至少不会悄悄算错。
broken = orders.copy()
broken["customer_id"] = broken["customer_id"].str.replace("C", "", regex=False).astype(int)
print("改完的类型:", broken["customer_id"].dtype, "| 值:", broken["customer_id"].head(3).tolist())

try:
    customers.merge(broken, on="customer_id", how="left")
except ValueError as e:
    print("报错:", str(e)[:120])

改完的类型: int64 | 值: [1, 2, 1]
报错: You are trying to merge on str and int64 columns for key 'customer_id'. If you wish to proceed you should use pd.concat


In [8]:
# 坑 B：类型都是字符串，但值不一样（"01" vs "C01"）→ 不报错，0 行匹配。这才是最坑的。
lookalike = orders.copy()
lookalike["customer_id"] = lookalike["customer_id"].str.replace("C", "", regex=False)
merged = customers.merge(lookalike, on="customer_id", how="left")
print("匹配上的行数:", merged["order_id"].notna().sum(), "（全空，但没有任何报错）")
merged[["customer_id", "order_id", "amount"]].head(3)

匹配上的行数: 0 （全空，但没有任何报错）


,customer_id,order_id,amount
0,C01,NaN,NaN
1,C02,NaN,NaN
2,C03,NaN,NaN


In [9]:
# 修复：把键统一成同一种"写法"，再 merge
fixed = lookalike.assign(customer_id="C" + lookalike["customer_id"])
print("修复后匹配行数:", customers.merge(fixed, on="customer_id", how="left")["order_id"].notna().sum())

修复后匹配行数: 9


In [10]:
# 列名不一致时用 left_on / right_on；两列都会保留，可以再 drop 掉重复的那一列
left = pd.DataFrame({"客户编号": ["C01", "C02"], "金额": [100, 200]})
right = pd.DataFrame({"customer_id": ["C01", "C03"], "level": ["金卡", "银卡"]})
left.merge(right, left_on="客户编号", right_on="customer_id", how="left")

,客户编号,金额,customer_id,level
0,C01,100,C01,金卡
1,C02,200,NaN,NaN


## 3. 多对多：行数会膨胀

如果两边在键上都有重复，`merge` 会做笛卡尔积：2 行 × 3 行 = 6 行。
聚合过的宽表和明细表一旦这样拼，指标就会被重复计算。

In [11]:
a = pd.DataFrame({"k": ["x", "x"], "v1": [1, 2]})
b = pd.DataFrame({"k": ["x", "x", "x"], "v2": [10, 20, 30]})
print("2 × 3 →", len(a.merge(b, on="k")), "行")
a.merge(b, on="k").head(6)

2 × 3 → 6 行


,k,v1,v2
0,x,1,10
1,x,1,20
2,x,1,30
3,x,2,10
4,x,2,20
5,x,2,30


## 4. `concat`：机械堆叠，不做匹配

- `axis=0`：上下拼（行变多），列名对齐，缺的列补 NaN；
- `axis=1`：左右拼（列变多），**按索引对齐**，索引对不上就补 NaN；
- `keys=`：加一层来源标记，适合合并多个月份的数据。

In [12]:
jan = pd.DataFrame({"city": ["北京", "上海"], "sales": [100, 200]})
feb = pd.DataFrame({"city": ["上海", "广州"], "sales": [150, 300]})

print("axis=0 上下堆叠（city 不做匹配，纯堆叠）:")
print(pd.concat([jan, feb], ignore_index=True))

print("\nkeys 标记来源:")
print(pd.concat([jan, feb], keys=["1月", "2月"], names=["month", "idx"]))

axis=0 上下堆叠（city 不做匹配，纯堆叠）:
   city  sales
0  北京    100
1  上海    200
2  上海    150
3  广州    300

keys 标记来源:
           city  sales
month idx             
1月   0    北京    100
      1    上海    200
2月   0    上海    150
      1    广州    300


In [13]:
# axis=1 按索引对齐：北京 只出现在 1 月，所以 2 月是 NaN
left = jan.set_index("city")
right = feb.set_index("city")
pd.concat([left, right], axis=1, keys=["1月", "2月"]).fillna(0)

,1月,2月
,sales,sales
city,,
北京,100.0,0.0
上海,200.0,150.0
广州,0.0,300.0


## 5. 宽表 ↔ 长表

| 方向 | 函数 | 什么时候用 |
| --- | --- | --- |
| 宽 → 长 | `melt` | 年份做成列了，想统一按"年份"分组 |
| 长 → 宽 | `pivot` | 想把"城市 × 品类"摊成矩阵 |
| 长 → 宽（带聚合） | `pivot_table` | 有重复值要先聚合 |
| 多层索引 ↔ 列 | `stack` / `unstack` | groupby 多列的结果转表格 |

In [14]:
prices_wide = pd.read_csv(DATA / "prices_wide.csv")
print("宽表：")
print(prices_wide)

long = prices_wide.melt(id_vars="product", var_name="year", value_name="price")
long["year"] = long["year"].astype(int)      # 列名转过来的年份是字符串
print("\n长表：")
print(long.sort_values(["product", "year"]).head(8))

宽表：
      product    2021    2022    2023    2024
0    智能音箱   299.0   279.0   259.0   249.0
1    无线耳机   499.0   469.0   429.0   399.0
2  扫地机器人  1999.0  1899.0  1799.0  1699.0
3    智能台灯   129.0   119.0   109.0    99.0

长表：
       product  year   price
2   扫地机器人  2021  1999.0
6   扫地机器人  2022  1899.0
10  扫地机器人  2023  1799.0
14  扫地机器人  2024  1699.0
1     无线耳机  2021   499.0
5     无线耳机  2022   469.0
9     无线耳机  2023   429.0
13    无线耳机  2024   399.0


In [15]:
# 长 → 宽：每个 product 一行，每年一列
long.pivot(index="product", columns="year", values="price").reset_index()

year,product,2021,2022,2023,2024
0,扫地机器人,1999.0,1899.0,1799.0,1699.0
1,无线耳机,499.0,469.0,429.0,399.0
2,智能台灯,129.0,119.0,109.0,99.0
3,智能音箱,299.0,279.0,259.0,249.0


In [16]:
# stack / unstack：Excel 里最像"逆透视"和"透视"的两个操作
wide = long.pivot(index="product", columns="year", values="price")
stacked = wide.stack()          # 列 → 行（长表，多层索引）
print("stack 后:")
print(stacked.head(6))
print("\nunstack 回来:")
print(stacked.unstack().iloc[:2, :3])

stack 后:
product     year
扫地机器人  2021    1999.0
            2022    1899.0
            2023    1799.0
            2024    1699.0
无线耳机    2021     499.0
            2022     469.0
dtype: float64

unstack 回来:
year          2021    2022    2023
product                           
扫地机器人  1999.0  1899.0  1799.0
无线耳机     499.0   469.0   429.0


## 6. 实战：三张表拼一张客户画像

目标：每个客户一行，包含基本信息、订单数、累计消费、客单价。

步骤：**订单先聚合（避免多对多膨胀）→ 再和客户表 left join**。

In [17]:
order_agg = orders.groupby("customer_id", as_index=False).agg(
    订单数=("order_id", "count"),
    累计消费=("amount", "sum"),
)
order_agg["客单价"] = (order_agg["累计消费"] / order_agg["订单数"]).round(1)

profile = customers.merge(order_agg, on="customer_id", how="left")
profile["订单数"] = profile["订单数"].fillna(0).astype(int)   # 没下过单的客户
profile["累计消费"] = profile["累计消费"].fillna(0.0)
profile.sort_values("累计消费", ascending=False)

,customer_id,name,city,level,signup_date,订单数,累计消费,客单价
0,C01,北京晨光科技,北京,金卡,2022-03-01,2,3897.0,1948.5
3,C04,深圳千里数码,深圳,金卡,2021-11-20,2,3268.7,1634.4
4,C05,杭州云栖家居,杭州,银卡,2023-05-06,1,599.0,599.0
1,C02,上海蓝鲸贸易,上海,银卡,2022-07-15,2,528.5,264.2
2,C03,广州南风服饰,广州,普通,2023-01-09,1,89.9,89.9
6,C07,北京朗月文化,北京,银卡,2024-01-02,1,45.5,45.5
5,C06,成都锦里食品,成都,普通,2023-08-18,0,0.0,NaN
7,C08,上海浦江物流,上海,普通,2022-09-30,0,0.0,NaN


## 7. 小练习

基于 `profile`：

1. 每个会员等级（`level`）的客户数、累计消费总额；
2. 有多少客户一单都没下过（`orders` 里匹配不到）；
3. 计算"复购客户占比"：订单数 ≥ 2 的客户数 / 下过单的客户数。

In [18]:
profile.groupby("level", as_index=False).agg(
    客户数=("customer_id", "count"),
    累计消费=("累计消费", "sum"),
).sort_values("累计消费", ascending=False)

,level,客户数,累计消费
1,金卡,2,7165.7
2,银卡,3,1173.0
0,普通,3,89.9


In [19]:
no_order = profile[profile["订单数"] == 0]
print("一单没下的客户数:", len(no_order), "|", no_order["name"].tolist())

一单没下的客户数: 2 | ['成都锦里食品', '上海浦江物流']


In [20]:
buyers = profile[profile["订单数"] >= 2]
print("复购客户数:", len(buyers), "/ 下过单的客户数:", (profile["订单数"] > 0).sum())
print("复购占比:", round(len(buyers) / (profile["订单数"] > 0).sum(), 2))

复购客户数: 3 / 下过单的客户数: 6
复购占比: 0.5


## 8. 小结与检查清单

**拼表之前**

- [ ] 连接键在两边**类型一致**吗？（`str` vs `int` 是头号杀手）
- [ ] 键值有没有空格、大小写、全角差异？
- [ ] 键在两侧**唯一**吗？不唯一就会膨胀，先 `validate="one_to_one"` 试一下。
- [ ] 拼完行数变化合理吗？对不上的是不是本该用 `left`？

**拼完之后**

- [ ] 用 `indicator=True` 看一遍未匹配的行；
- [ ] 用 `df.shape` 和 `df["键"].nunique()` 验证没有意外膨胀；
- [ ] `axis=1` 的 `concat` 是按索引对齐的，先确认索引含义一致。

**一句话总结**：`merge` 认列、`join` 认索引、`concat` 只堆叠；宽长转换记住 `melt` 与 `pivot` 互为逆操作。